# Manchu CRNN from scratch

A PyTorch CNN + bidirectional LSTM + CTC recognizer that predicts romanized characters. Training uses the workspace's generated word images; validation includes a separately sourced, attributed 500-image tyotakuki sample. These are synthetic images, not historical documents.

In Colab choose Runtime > Change runtime type > GPU, then run the cells in order. Set RUN_CRNN_TRAINING = True to train. Drive will ask you to authorize mounting. This creates a new model with random weights. Training is opt-in; checkpoints and reports are written inside the dedicated Drive folder. Review validation results before making architecture changes. The tyotakuki sample is a development set, not a final held-out test set.

In [ ]:
from pathlib import Path
import csv
import hashlib
import io
import json
import math
import platform
import random
import unicodedata
import zipfile
from collections import Counter
from datetime import datetime, timezone

from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/4 Senior/Thesis")
WORKSPACE = DRIVE / "Manchu-CRNN-Colab"
DATA = DRIVE / "data"
MODELS = DRIVE / "models"
RUN = WORKSPACE / "runs" / "crnn_v1"
BENCHMARK_CSV = WORKSPACE / "tyotakuki_validation.csv"
IMAGE_ZIP = DATA / "images" / "images_new.zip"
WORDS_CSV = DATA / "images" / "words_new.csv"
CNN_CHECKPOINT = MODELS / "resnet18_best.pt"
CRNN_CHECKPOINT = RUN / "crnn_best.pt"
SEED = 42

CREATE_ANNOTATION_TEMPLATE = False
BENCHMARK_REVIEWED = False
REAL_EVALUATION_SPLIT = "val"  # Change to "test" only after fixing the protocol.
CNN_CLASS_ORDER_CONFIRMED = False
TRUST_LEGACY_CNN_PICKLE = False  # Old full-model files require this explicit trust decision.
LOAD_CNN = False
RUN_CNN_EVALUATION = False
RUN_CRNN_TRAINING = False  # Set True to train; choose a new RUN for each experiment.
LOAD_SAVED_CRNN = False
RUN_CRNN_EVALUATION = False
SAVE_REPORTS = False
RUN_SMOKE_TESTS = True  # Tiny random tensors only; no optimizer updates or real data.

# The existing generated images are horizontal. Inspect real crops and set each rotation.
REAL_ROTATION_ALLOWED = {0, 90, 180, 270}  # Counterclockwise, in degrees.
IMAGE_HEIGHT = 32
MAX_SEQUENCE_WIDTH = 1024
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-3
VAL_FONTS = {"XM_WenQin"}
TEST_FONTS = {"XM_YaBai", "XM_YingBi"}

for name, path in {"images": IMAGE_ZIP, "labels": WORDS_CSV,
                   "CNN": CNN_CHECKPOINT, "annotations": BENCHMARK_CSV}.items():
    print(f"{name}: {path} (exists={path.exists()})")
print("Local outputs:", RUN)

## 1. Load the vocabulary and inspect the generated dataset

Images are read directly from the existing ZIP; identical vocabulary rows are collapsed in memory.

In [ ]:
def normalize_text(text):
    # Preserve case, punctuation, diacritics, and word boundaries; NFC only.
    return unicodedata.normalize("NFC", text.strip())

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def local_output(path):
    path = Path(path).resolve()
    path.relative_to(RUN.resolve())
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

def write_json(path, value):
    local_output(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n")

with WORDS_CSV.open(newline="", encoding="utf-8-sig") as stream:
    vocabulary_rows = list(csv.DictReader(stream))
roman_to_manchu = {}
identical_duplicates = 0
for row in vocabulary_rows:
    roman = normalize_text(row["romanization"])
    if not roman:
        raise ValueError("Empty romanization in vocabulary.")
    if roman in roman_to_manchu:
        if roman_to_manchu[roman] != row["manchu"]:
            raise ValueError(f"Conflicting Manchu spellings for {roman!r}; review the labels.")
        identical_duplicates += 1
    roman_to_manchu[roman] = row["manchu"]
print("Identical vocabulary duplicates collapsed in memory:", identical_duplicates)

# Inspect ZIP metadata without extracting or loading image pixels.
with zipfile.ZipFile(IMAGE_ZIP) as archive:
    synthetic = []
    for member in archive.namelist():
        p = Path(member)
        if "__MACOSX" in p.parts or p.suffix.lower() != ".png":
            continue
        roman = normalize_text(p.parent.name)
        prefix = p.parent.name + "_"
        if not p.stem.startswith(prefix):
            raise ValueError(f"Unexpected image naming: {member}")
        synthetic.append({"member": member, "romanization": roman,
                          "font": p.stem[len(prefix):]})
archive_classes = sorted({r["romanization"] for r in synthetic})
if set(archive_classes) != set(roman_to_manchu):
    raise ValueError("Archive classes and words_new.csv do not match; choose a matched pair.")
# torchvision ImageFolder sorts class-directory names; this is a candidate mapping,
# not proof that a particular checkpoint was trained on this archive.
cnn_classes = archive_classes
print(f"{len(synthetic)} images, {len(cnn_classes)} word classes")
print("Fonts:", sorted({r["font"] for r in synthetic}))
print("Candidate CNN class order:", cnn_classes[:10])

## 2. Load the external synthetic validation sample

Labels are provided by tyotakuki and lowercased for this experiment. Inspect image orientation and transcription conventions before interpreting scores. The sample is for development, and its labels are not verified historical transcriptions.

In [ ]:
ANNOTATION_FIELDS = ["sample_id", "page_id", "image_path", "x0", "y0", "x1", "y1",
                     "rotate_ccw", "romanization", "split"]
if CREATE_ANNOTATION_TEMPLATE:
    path = local_output(BENCHMARK_CSV)
    with path.open("x", newline="", encoding="utf-8") as stream:
        csv.DictWriter(stream, fieldnames=ANNOTATION_FIELDS).writeheader()
    print("Created blank annotation template:", path)

def read_benchmark(path):
    if not Path(path).exists():
        return []
    with Path(path).open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if not set(ANNOTATION_FIELDS).issubset(reader.fieldnames or []):
            raise ValueError("Annotation CSV is missing required columns.")
        rows = list(reader)
    identifiers, regions, page_splits, image_pages = set(), set(), {}, {}
    for row in rows:
        if not row["sample_id"] or row["sample_id"] in identifiers:
            raise ValueError("Each sample needs a unique nonempty sample_id.")
        identifiers.add(row["sample_id"])
        row["romanization"] = normalize_text(row["romanization"])
        if not row["romanization"] or not row["page_id"]:
            raise ValueError("Every sample needs a checked transcription and source page_id.")
        if row["split"] not in {"val", "test"}:
            raise ValueError("Real benchmark split must be val or test.")
        previous = page_splits.setdefault(row["page_id"], row["split"])
        if previous != row["split"]:
            raise ValueError("The same source page appears in validation and test.")
        image_path = Path(row["image_path"])
        row["path"] = (image_path if image_path.is_absolute() else WORKSPACE / image_path).resolve()
        if not row["path"].is_file():
            raise FileNotFoundError(row["path"])
        external = (DRIVE / "tyotakuki").resolve()
        if row["path"] == external or external in row["path"].parents:
            raise ValueError("tyotakuki material is excluded from this benchmark.")
        if image_pages.setdefault(str(row["path"]), row["page_id"]) != row["page_id"]:
            raise ValueError("One image is assigned multiple source page IDs.")
        box = [row[k].strip() for k in ("x0", "y0", "x1", "y1")]
        if any(box) and not all(box):
            raise ValueError("Provide all four crop coordinates or none.")
        row["box"] = tuple(map(int, box)) if all(box) else None
        row["rotate_ccw"] = int(row["rotate_ccw"] or 0)
        if row["rotate_ccw"] not in REAL_ROTATION_ALLOWED:
            raise ValueError("Rotation must be 0, 90, 180, or 270 degrees.")
        region = (str(row["path"]), row["box"])
        if region in regions:
            raise ValueError("Duplicate image region in benchmark.")
        regions.add(region)
    return rows

real_rows = read_benchmark(BENCHMARK_CSV)
print("Real crops:", len(real_rows), Counter(r["split"] for r in real_rows))
if not real_rows:
    print("Next action: create the template, transcribe word crops, then rerun this cell.")

In [ ]:
import importlib.metadata
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchvision import models, transforms

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
# CPU is the portable default; use CUDA when available. CTC MPS support varies,
# so this baseline deliberately uses CPU on Apple Silicon.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERSIONS = {name: importlib.metadata.version(name)
            for name in ["torch", "torchvision", "Pillow", "numpy"]}
print("Device:", DEVICE, "Versions:", VERSIONS)

cnn_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BILINEAR,
                      antialias=True),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3),
])

def real_crop(row):
    with Image.open(row["path"]) as source:
        image = ImageOps.exif_transpose(source).convert("L")
    if row["box"] is not None:
        x0, y0, x1, y1 = row["box"]
        if not (0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height):
            raise ValueError(f"Invalid crop for {row['sample_id']}: {row['box']}")
        image = image.crop(row["box"])
    image = image.rotate(row["rotate_ccw"], expand=True, fillcolor=255)
    # Match the black-on-white synthetic images used to train the CNN and CRNN.
    if np.asarray(image).mean() < 127:
        image = ImageOps.invert(image)
    return image

def zip_image(member):
    with zipfile.ZipFile(IMAGE_ZIP) as archive:
        contents = archive.read(member)
    with Image.open(io.BytesIO(contents)) as image:
        return image.convert("L")

def show_benchmark(rows, limit=8):
    if not rows:
        print("No annotated crops yet.")
        return
    chosen = rows[:limit]
    fig, axes = plt.subplots(len(chosen), 1, figsize=(10, 2 * len(chosen)), squeeze=False)
    for axis, row in zip(axes[:, 0], chosen):
        axis.imshow(real_crop(row), cmap="gray", vmin=0, vmax=255)
        axis.set_title(f"{row['sample_id']} | {row['romanization']} | {row['split']}")
        axis.axis("off")
    plt.tight_layout()

# Inspect validation crops first; avoid browsing held-out test pages during tuning.
show_benchmark([r for r in real_rows if r["split"] == "val"])

## 3. Define word and character metrics

Character error rate counts insertions, deletions and substitutions divided by reference characters. Word accuracy requires an exact match.

In [ ]:
def edit_distance(reference, prediction):
    previous = list(range(len(prediction) + 1))
    for i, expected in enumerate(reference, 1):
        current = [i]
        for j, observed in enumerate(prediction, 1):
            current.append(min(current[-1] + 1, previous[j] + 1,
                               previous[j - 1] + (expected != observed)))
        previous = current
    return previous[-1]

def summarize_predictions(rows):
    if not rows:
        return {"n": 0, "word_accuracy": None, "word_error_rate": None,
                "cer": None, "cnn_vocabulary_coverage": None}
    exact = sum(r["reference"] == r["prediction"] for r in rows)
    edits = sum(edit_distance(r["reference"], r["prediction"]) for r in rows)
    characters = sum(len(r["reference"]) for r in rows)
    return {"n": len(rows), "word_accuracy": exact / len(rows),
            "word_error_rate": 1 - exact / len(rows), "cer": edits / characters,
            "cnn_vocabulary_coverage": sum(r["in_cnn_vocabulary"] for r in rows) / len(rows)}

def benchmark_slices(rows):
    result = {}
    for split in ("val", "test"):
        selected = [r for r in rows if r["split"] == split]
        result[split] = {"all": summarize_predictions(selected),
                         "in_vocabulary": summarize_predictions([r for r in selected if r["in_cnn_vocabulary"]]),
                         "out_of_vocabulary": summarize_predictions([r for r in selected if not r["in_cnn_vocabulary"]])}
    return result

assert edit_distance("", "a") == 1
assert edit_distance("manju", "manju") == 0
assert edit_distance("manju", "manu") == 1
assert normalize_text(" e\u0301 ") == "é"
print("Metric checks passed.")

## 4. Split synthetic training data by word and font

Training, validation, and synthetic test splits use distinct word/font combinations. The tyotakuki sample is separate from these splits. The alphabet is declared from the source vocabulary; missing characters are reported.

In [ ]:
all_words = sorted(roman_to_manchu)
shuffled_words = all_words.copy()
random.Random(SEED).shuffle(shuffled_words)
n_val = max(1, len(all_words) // 10)
n_test = max(1, len(all_words) // 10)
val_words = set(shuffled_words[:n_val])
test_words = set(shuffled_words[n_val:n_val + n_test])
train_words = set(shuffled_words[n_val + n_test:])
all_fonts = {r["font"] for r in synthetic}
assert VAL_FONTS.isdisjoint(TEST_FONTS)
assert VAL_FONTS | TEST_FONTS <= all_fonts
train_fonts = all_fonts - VAL_FONTS - TEST_FONTS
assert train_fonts and train_words and val_words and test_words

synthetic_splits = {"train": [], "val": [], "test_seen_words": [], "test_unseen_words": []}
for record in synthetic:
    word, font = record["romanization"], record["font"]
    if word in train_words and font in train_fonts:
        synthetic_splits["train"].append(record)
    elif word in val_words and font in VAL_FONTS:
        synthetic_splits["val"].append(record)
    elif font in TEST_FONTS and word in train_words:
        synthetic_splits["test_seen_words"].append(record)
    elif font in TEST_FONTS and word in test_words:
        synthetic_splits["test_unseen_words"].append(record)
assert all(synthetic_splits.values())
char_to_id = {char: i + 1 for i, char in enumerate(sorted(set("".join(all_words))))}
id_to_char = {i: char for char, i in char_to_id.items()}
BLANK = 0  # CTC blank is not a transcription character.
print("Synthetic split sizes:", {k: len(v) for k, v in synthetic_splits.items()})
print("Alphabet:", "".join(char_to_id))
print("Unknown characters in real annotations:",
      sorted(set("".join(r["romanization"] for r in real_rows)) - set(char_to_id)))

## 5. Build the CRNN and CTC decoder

The convolutional layers extract features, the bidirectional LSTM models the sequence, and CTC learns character alignments. Variable image widths are handled individually and packed before the LSTM.

In [ ]:
def sequence_tensor(image):
    image = image.convert("L")
    width = max(64, round(image.width * IMAGE_HEIGHT / image.height))
    if width > MAX_SEQUENCE_WIDTH:
        raise ValueError(f"Sequence width {width} exceeds {MAX_SEQUENCE_WIDTH}; inspect the crop.")
    image = image.resize((width, IMAGE_HEIGHT), Image.Resampling.BILINEAR)
    return transforms.ToTensor()(image).sub(0.5).div(0.5)

class SyntheticWords(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        return sequence_tensor(zip_image(row["member"])), row["romanization"]

def collate_words(batch):
    images, texts = zip(*batch)
    widths = torch.tensor([image.shape[-1] for image in images], dtype=torch.long)
    padded = torch.ones(len(images), 1, IMAGE_HEIGHT, int(widths.max()))
    for i, image in enumerate(images):
        padded[i, :, :, :image.shape[-1]] = image
    encoded = []
    for text, width in zip(texts, widths.tolist()):
        unknown = set(text) - set(char_to_id)
        if unknown:
            raise ValueError(f"Characters outside the declared alphabet: {unknown}")
        minimum_steps = len(text) + sum(a == b for a, b in zip(text, text[1:]))
        if not text or width // 4 < minimum_steps:
            raise ValueError(f"CTC target cannot fit: {text!r}, width={width}.")
        encoded.append(torch.tensor([char_to_id[c] for c in text], dtype=torch.long))
    return (padded, widths, torch.cat(encoded),
            torch.tensor([len(t) for t in encoded], dtype=torch.long), list(texts))

class CRNN(nn.Module):
    def __init__(self, num_characters):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
        )
        self.recurrent = nn.LSTM(128, 128, num_layers=2, batch_first=True,
                                 bidirectional=True, dropout=0.1)
        self.classifier = nn.Linear(256, num_characters)

    def forward(self, images, widths):
        # Crop each image before the CNN so batch padding cannot leak through convolutions.
        # This simple baseline favors correctness; bucket widths for speed later.
        features = [self.features(image[:, :, :int(width)].unsqueeze(0))
                    .mean(dim=2).squeeze(0).transpose(0, 1)
                    for image, width in zip(images, widths.tolist())]
        lengths = torch.div(widths, 4, rounding_mode="floor").cpu()
        sequence = nn.utils.rnn.pad_sequence(features, batch_first=True)
        packed = pack_padded_sequence(sequence, lengths, batch_first=True, enforce_sorted=False)
        recurrent, _ = self.recurrent(packed)
        recurrent, _ = pad_packed_sequence(recurrent, batch_first=True)
        logits = self.classifier(recurrent).transpose(0, 1)  # time × batch × alphabet
        return logits.log_softmax(dim=-1), lengths

def collapse_ctc(tokens):
    result, previous = [], None
    for token in tokens:
        if token != previous and token != BLANK:
            result.append(id_to_char[token])
        previous = token  # A blank separates two copies of the same character.
    return "".join(result)

def greedy_decode(log_probs, lengths):
    tokens = log_probs.argmax(dim=-1).detach().cpu()
    return [collapse_ctc(tokens[:int(length), i].tolist()) for i, length in enumerate(lengths)]

crnn = CRNN(len(char_to_id) + 1).to(DEVICE)
crnn_ready = False
print("New CRNN parameters:", sum(p.numel() for p in crnn.parameters()))

In [ ]:
if RUN_SMOKE_TESTS:
    a = char_to_id["a"]
    assert collapse_ctc([a, a, BLANK, a]) == "aa"
    small = CRNN(len(char_to_id) + 1).cpu().eval()
    fake_batch = [(torch.zeros(1, IMAGE_HEIGHT, 64), "a"),
                  (torch.ones(1, IMAGE_HEIGHT, 80), "aa")]
    images, widths, targets, target_lengths, texts = collate_words(fake_batch)
    with torch.no_grad():
        log_probs, lengths = small(images, widths)
        loss = nn.CTCLoss(blank=BLANK, zero_infinity=False)(
            log_probs, targets, lengths, target_lengths)
    assert tuple(log_probs.shape) == (20, 2, len(char_to_id) + 1)
    assert lengths.tolist() == [16, 20] and torch.isfinite(loss)
    assert len(greedy_decode(log_probs, lengths)) == 2
    # A crop must give the same logits alone and in a padded batch.
    with torch.no_grad():
        alone, _ = small(images[:1, :, :, :64], widths[:1])
    torch.testing.assert_close(log_probs[:16, :1], alone, rtol=1e-4, atol=1e-5)
    print("Passed: CTC repeats, variable widths, padding invariance, and finite CTC loss.")
    del small

## 6. Check a generated input

Confirm the rendered word reads left to right after rotation, matching the sample validation images.

In [ ]:
# Read only one synthetic image for visual inspection; do not render a full dataset.
example_record = synthetic_splits["train"][0]
example_image = zip_image(example_record["member"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].imshow(example_image, cmap="gray")
axes[0].set_title(example_record["romanization"] + " — source")
axes[1].imshow(sequence_tensor(example_image)[0], cmap="gray", vmin=-1, vmax=1)
axes[1].set_title("CRNN input")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

## 7. Train the baseline

Set `RUN_CRNN_TRAINING = True` in the configuration and rerun from the top. Training saves a local checkpoint, split manifest, configuration, and history; it refuses to overwrite an existing run. Best checkpoint selection uses synthetic validation CER. Training time depends on hardware and has not been established.

In [ ]:
def make_loader(records, shuffle=False):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(SyntheticWords(records), batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=collate_words, num_workers=0, generator=generator)

@torch.inference_mode()
def evaluate_synthetic(model, records):
    model.eval()
    rows = []
    for images, widths, targets, target_lengths, texts in make_loader(records):
        log_probs, lengths = model(images.to(DEVICE), widths)
        predicted = greedy_decode(log_probs, lengths)
        rows.extend({"reference": ref, "prediction": pred,
                     "in_cnn_vocabulary": ref in roman_to_manchu}
                    for ref, pred in zip(texts, predicted))
    return summarize_predictions(rows)

def fit_crnn(model):
    if CRNN_CHECKPOINT.exists() or (RUN / "split_manifest.json").exists():
        raise FileExistsError("This run has training artifacts. Choose a new RUN directory.")
    configuration = {"seed": SEED, "image_height": IMAGE_HEIGHT,
                     "max_sequence_width": MAX_SEQUENCE_WIDTH, "batch_size": BATCH_SIZE,
                     "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
                     "versions": VERSIONS, "python": platform.python_version(),
                     "image_zip": str(IMAGE_ZIP), "words_csv": str(WORDS_CSV),
                     "words_sha256": sha256_file(WORDS_CSV),
                     "image_zip_sha256": sha256_file(IMAGE_ZIP),
                     "train_fonts": sorted(train_fonts), "val_fonts": sorted(VAL_FONTS),
                     "test_fonts": sorted(TEST_FONTS), "char_to_id": char_to_id,
                     "started_utc": datetime.now(timezone.utc).isoformat()}
    write_json(RUN / "split_manifest.json", synthetic_splits)
    write_json(RUN / "config.json", configuration)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CTCLoss(blank=BLANK, zero_infinity=False)
    best_cer, history = float("inf"), []
    loader = make_loader(synthetic_splits["train"], shuffle=True)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, samples = 0.0, 0
        for images, widths, targets, target_lengths, texts in loader:
            optimizer.zero_grad(set_to_none=True)
            log_probs, lengths = model(images.to(DEVICE), widths)
            # CPU CTC avoids backend-specific target/length restrictions.
            loss = criterion(log_probs.cpu(), targets, lengths, target_lengths)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite CTC loss; inspect sequence lengths and labels.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item() * len(texts)
            samples += len(texts)
        validation = evaluate_synthetic(model, synthetic_splits["val"])
        item = {"epoch": epoch, "train_loss": total_loss / samples, "val": validation}
        history.append(item)
        write_json(RUN / "history.json", history)
        print(json.dumps(item))
        if validation["cer"] < best_cer:
            best_cer = validation["cer"]
            # Plain metadata plus CPU tensors supports restricted weights-only loading.
            state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            torch.save({"model_state_dict": state, "char_to_id": char_to_id,
                        "config": configuration, "epoch": epoch, "val_cer": best_cer},
                       local_output(CRNN_CHECKPOINT))
    return history

history = None
if RUN_CRNN_TRAINING:
    if EPOCHS < 1:
        raise ValueError("EPOCHS must be positive.")
    history = fit_crnn(crnn)
else:
    print("Training disabled. Inspect inputs and review the split before enabling it.")

## 8. Load the best checkpoint

To resume evaluation later, set `LOAD_SAVED_CRNN = True` and keep the same settings and source files. A newly initialized model is never evaluated as a trained model.

In [ ]:
if LOAD_SAVED_CRNN or RUN_CRNN_TRAINING:
    checkpoint = torch.load(CRNN_CHECKPOINT, map_location="cpu", weights_only=True)
    if checkpoint["char_to_id"] != char_to_id:
        raise ValueError("Checkpoint alphabet differs from this notebook's alphabet.")
    config = checkpoint["config"]
    if (config["image_height"], config["max_sequence_width"]) != (IMAGE_HEIGHT, MAX_SEQUENCE_WIDTH):
        raise ValueError("Checkpoint image preprocessing differs from this configuration.")
    if config["words_sha256"] != sha256_file(WORDS_CSV):
        raise ValueError("Vocabulary changed since training.")
    if config["image_zip_sha256"] != sha256_file(IMAGE_ZIP):
        raise ValueError("Image archive changed since training.")
    saved_splits = json.loads((RUN / "split_manifest.json").read_text())
    if saved_splits != synthetic_splits:
        raise ValueError("Current synthetic split differs from the saved experiment.")
    crnn.load_state_dict(checkpoint["model_state_dict"], strict=True)
    crnn = crnn.to(DEVICE).eval()
    crnn_ready = True
    print("Loaded CRNN epoch", checkpoint["epoch"], "validation CER", checkpoint["val_cer"])
else:
    print("No trained CRNN loaded; random weights will not be reported as a baseline result.")

crnn_predictions = []
if RUN_CRNN_EVALUATION:
    if not crnn_ready:
        raise RuntimeError("Train or load the selected CRNN first.")
    crnn_predictions = evaluate_real(crnn, "crnn", real_rows)
    print(json.dumps(benchmark_slices(crnn_predictions), indent=2))

## 9. Evaluate tyotakuki validation images

The original labels are lowercased. Dark backgrounds are inverted to match the training images. Report character error rate and exact word accuracy, and inspect examples before drawing conclusions. The evaluation set has not been used to select checkpoints in this notebook.

In [ ]:
@torch.inference_mode()
def evaluate_tyotakuki(model, rows):
    model.eval()
    results = []
    for row in rows:
        image = sequence_tensor(real_crop(row))
        widths = torch.tensor([image.shape[-1]], dtype=torch.long)
        log_probs, lengths = model(image.unsqueeze(0).to(DEVICE), widths)
        prediction = normalize_text(greedy_decode(log_probs, lengths)[0])
        results.append({"sample_id": row["sample_id"], "reference": row["romanization"],
                        "prediction": prediction,
                        "in_cnn_vocabulary": row["romanization"] in roman_to_manchu})
    return results

if crnn_ready:
    results = evaluate_tyotakuki(crnn, real_rows)
    metrics = summarize_predictions(results)
    print(json.dumps(metrics, indent=2))
    for r in results[:20]:
        print(r["sample_id"], repr(r["reference"]), "->", repr(r["prediction"]))
    write_json(RUN / "tyotakuki_validation.json", {
        "source": "tyotakuki, 500-image evaluation sample",
        "annotations_sha256": sha256_file(BENCHMARK_CSV),
        "checkpoint_sha256": sha256_file(CRNN_CHECKPOINT),
        "metrics": metrics, "predictions": results})
else:
    print("Train or load a CRNN before evaluating; no model results yet.")

## Next steps

Compare character errors with the ResNet baseline, inspect label conventions and domain differences, and evaluate unseen words separately. Preserve tyotakuki attribution and keep the final test set independent of training and tuning. Real-document evaluation remains a later step.

References: [CRNN paper](https://arxiv.org/abs/1507.05717) and [PyTorch CTC loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CTCLoss.html).